# KVO_01 — MOS_1_2 pixel-wise Musanga U-Net

Clean single-mosaic test following the original Musanga NB01–02 workflow: rasterize classified crowns, train a **pixel-wise U-Net (ResNet18 encoder)**, run full-resolution probability inference, and evaluate the held-out test data at **both pixel and crown level**.

This notebook deliberately stops at the UAV product. Satellite aggregation/modeling comes later. The structure is ready to extend to additional mosaics later.

In [ ]:
!pip -q install geopandas rasterio pyogrio segmentation-models-pytorch scikit-learn matplotlib tqdm

from pathlib import Path
import os, json, math, random, shutil, time, hashlib, warnings
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve, confusion_matrix
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from google.colab import drive
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark=True
print("DEVICE:",DEVICE)
if DEVICE=="cuda": print(torch.cuda.get_device_name(0))

In [ ]:
DRIVE_ROOT=Path("/content/drive")
MYDRIVE=DRIVE_ROOT/"MyDrive"
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()): drive.mount(str(DRIVE_ROOT), force_remount=False)
ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"
MOSAIC=ROOT/"data"/"Geo_Ref"/"MOS_1_2_kvo.tif"
ANNOT=ROOT/"data"/"Geo_Ref"/"Annotations"/"Tree_Annotations_MOS_1_2_kvo.shp"
OUT=ROOT/"KVO_work"/"MOS_1_2_pixel_unet"
MODEL_DIR=OUT/"model"; EVAL_DIR=OUT/"evaluation"; PROB_DIR=OUT/"probability"; CACHE_DRIVE=OUT/"cache"
for p in [OUT,MODEL_DIR,EVAL_DIR,PROB_DIR,CACHE_DRIVE]: p.mkdir(parents=True,exist_ok=True)
assert MOSAIC.exists(), MOSAIC
assert ANNOT.exists(), ANNOT
print(MOSAIC); print(ANNOT); print("OUT:",OUT)

## 1 — Audit and labels

`Class_ID == 1` is Musanga. Other classified crowns are non-Musanga. **Unlabelled canopy/background is ignored during loss and evaluation**, rather than silently treated as a known non-Musanga tree.

In [ ]:
with rasterio.open(MOSAIC) as src:
    CRS=src.crs; TRANSFORM=src.transform; W,H=src.width,src.height
    GSD=(abs(src.transform.a),abs(src.transform.e)); BOUNDS=src.bounds; DTYPE=src.dtypes[0]
print({"size":(W,H),"bands":rasterio.open(MOSAIC).count,"crs":str(CRS),"gsd":GSD,"dtype":DTYPE})
ann=gpd.read_file(ANNOT)
if ann.crs!=CRS: ann=ann.to_crs(CRS)
ann=ann[ann.geometry.notna() & ~ann.geometry.is_empty & ann["Class_ID"].notna()].copy()
ann["class_id_num"]=pd.to_numeric(ann["Class_ID"],errors="coerce")
ann=ann[ann.class_id_num.notna()].copy(); ann["class_id_num"]=ann.class_id_num.astype(int)
ann["target"]=(ann.class_id_num==1).astype(np.uint8)
ann["crown_id"]=np.arange(len(ann),dtype=np.int64)
print(ann.groupby(["target","class_id_num"]).size())
print("Musanga:",int(ann.target.sum()),"Other:",int((1-ann.target).sum()),"Total:",len(ann))

## 2 — Spatial train / validation / test split

Crowns are assigned by centroid to spatial blocks, so the test set is spatially held out rather than a random collection of neighbouring crowns. Blocks are assigned 70/15/15; the cell retries deterministic seeds until all splits contain both classes.

In [ ]:
BLOCK_M=100.0
c=ann.geometry.centroid
ann["bx"]=np.floor((c.x-BOUNDS.left)/BLOCK_M).astype(int)
ann["by"]=np.floor((c.y-BOUNDS.bottom)/BLOCK_M).astype(int)
ann["block_id"]=ann.bx.astype(str)+"_"+ann.by.astype(str)
blocks=ann.block_id.unique()

def make_split(seed):
    rng=np.random.default_rng(seed); b=blocks.copy(); rng.shuffle(b)
    n=len(b); ntr=max(1,int(.70*n)); nv=max(1,int(.15*n))
    mp={x:"train" for x in b[:ntr]}; mp.update({x:"val" for x in b[ntr:ntr+nv]}); mp.update({x:"test" for x in b[ntr+nv:]})
    s=ann.block_id.map(mp)
    ok=all(ann.loc[s==q,"target"].nunique()==2 for q in ["train","val","test"])
    return s,ok
for k in range(1000):
    s,ok=make_split(SEED+k)
    if ok: break
assert ok,"Could not create spatial split containing both classes."
ann["split"]=s
print("block seed:",SEED+k,"blocks:",ann.block_id.nunique())
print(ann.groupby(["split","target"]).size())
ann.to_file(OUT/"MOS_1_2_crowns_spatial_split.gpkg",layer="crowns",driver="GPKG")

## 3 — Local SSD + split-aware raster masks + persistent patch cache

The large source TIFF is copied once to Colab SSD. Training examples are fixed 256×256 raster patches containing labelled crown pixels. Label coding is **1 = Musanga, 0 = other classified crown, 255 = ignore**. The cache is fingerprinted and persisted on Drive, so later sessions can restore the much smaller training cache instead of repeatedly extracting patches.

In [ ]:
LOCAL=Path("/content/musanga_kvo01"); LOCAL.mkdir(parents=True,exist_ok=True)
LOCAL_MOSAIC=LOCAL/MOSAIC.name
PATCH=256; STRIDE=256; IGNORE=255
fp_payload={"mosaic_size":MOSAIC.stat().st_size,"annot_size":ANNOT.stat().st_size,"patch":PATCH,"stride":STRIDE,"block_m":BLOCK_M,"seed":SEED+k}
FP=hashlib.sha1(json.dumps(fp_payload,sort_keys=True).encode()).hexdigest()[:12]
DRIVE_NPZ=CACHE_DRIVE/f"patches_{FP}.npz"; LOCAL_NPZ=LOCAL/f"patches_{FP}.npz"

if DRIVE_NPZ.exists():
    print("Restoring persistent patch cache:",DRIVE_NPZ)
    shutil.copy2(DRIVE_NPZ,LOCAL_NPZ)
else:
    if not LOCAL_MOSAIC.exists() or LOCAL_MOSAIC.stat().st_size!=MOSAIC.stat().st_size:
        free=shutil.disk_usage("/content").free; need=MOSAIC.stat().st_size
        assert free>need*1.15, f"Need ~{need/1e9:.1f} GB free for local mosaic copy."
        print(f"Copying {need/1e9:.2f} GB mosaic to local SSD..."); t=time.time(); shutil.copy2(MOSAIC,LOCAL_MOSAIC); print(f"{(time.time()-t)/60:.1f} min")
    else: print("Reusing local mosaic.")

    # Rasterize each split separately. Pixels outside classified crowns stay IGNORE.
    masks={}
    for split in ["train","val","test"]:
        a=ann[ann.split==split]
        shapes=[(g,int(y)) for g,y in zip(a.geometry,a.target)]
        masks[split]=rasterize(shapes,out_shape=(H,W),transform=TRANSFORM,fill=IGNORE,dtype="uint8",all_touched=False)

    imgs=[]; ys=[]; splits=[]; wins=[]
    with rasterio.open(LOCAL_MOSAIC) as src:
        for r0 in tqdm(range(0,H,PATCH),desc="Building labelled patch cache"):
            for c0 in range(0,W,PATCH):
                hh=min(PATCH,H-r0); ww=min(PATCH,W-c0)
                # Assign a patch only when it contains labels from exactly one split.
                present=[]
                for sp in ["train","val","test"]:
                    if np.any(masks[sp][r0:r0+hh,c0:c0+ww]!=IGNORE): present.append(sp)
                if len(present)!=1: continue
                sp=present[0]; m=masks[sp][r0:r0+hh,c0:c0+ww]
                if not np.any(m!=IGNORE): continue
                arr=src.read([1,2,3],window=Window(c0,r0,ww,hh),boundless=True,fill_value=0,out_shape=(3,PATCH,PATCH),resampling=rasterio.enums.Resampling.bilinear)
                mm=np.full((PATCH,PATCH),IGNORE,np.uint8); mm[:hh,:ww]=m
                imgs.append(arr.astype(np.uint8)); ys.append(mm); splits.append({"train":0,"val":1,"test":2}[sp]); wins.append((r0,c0,hh,ww))
    X=np.stack(imgs); Y=np.stack(ys); S=np.asarray(splits,np.uint8); WIN=np.asarray(wins,np.int32)
    np.savez_compressed(LOCAL_NPZ,X=X,Y=Y,S=S,WIN=WIN)
    shutil.copy2(LOCAL_NPZ,DRIVE_NPZ)
    del masks
    print("Persistent cache saved:",DRIVE_NPZ)

d=np.load(LOCAL_NPZ); X=d["X"]; Y=d["Y"]; S=d["S"]; WIN=d["WIN"]
print("patches train/val/test:",[(S==i).sum() for i in range(3)],"RAM:",round((X.nbytes+Y.nbytes)/1e6,1),"MB")

## 4 — U-Net training

Same model family as the earlier Musanga workflow: U-Net with ImageNet-pretrained ResNet18 encoder. BCE loss is calculated **only on labelled crown pixels**. Validation AP selects the checkpoint. L4/T4 uses mixed precision.

In [ ]:
MEAN=np.array([0.485,0.456,0.406],np.float32)[:,None,None]
STD=np.array([0.229,0.224,0.225],np.float32)[:,None,None]
class PatchDS(Dataset):
    def __init__(self,code,augment=False): self.idx=np.where(S==code)[0]; self.augment=augment
    def __len__(self): return len(self.idx)
    def __getitem__(self,j):
        i=self.idx[j]; x=X[i].astype(np.float32)/255.; y=Y[i].copy()
        if self.augment:
            if np.random.rand()<.5: x=x[:,:,::-1].copy(); y=y[:,::-1].copy()
            if np.random.rand()<.5: x=x[:,::-1,:].copy(); y=y[::-1,:].copy()
            k=np.random.randint(4); x=np.rot90(x,k,axes=(1,2)).copy(); y=np.rot90(y,k).copy()
            x=np.clip(x*np.random.uniform(.9,1.1),0,1)
        x=(x-MEAN)/STD
        return torch.from_numpy(x),torch.from_numpy(y.astype(np.int64))

def loader(code,train=False):
    ds=PatchDS(code,train)
    return DataLoader(ds,batch_size=16 if DEVICE=="cuda" else 4,shuffle=train,num_workers=2,pin_memory=(DEVICE=="cuda"),persistent_workers=True,prefetch_factor=2)
train_dl,val_dl,test_dl=loader(0,True),loader(1),loader(2)
print("batches:",len(train_dl),len(val_dl),len(test_dl))

model=smp.Unet(encoder_name="resnet18",encoder_weights="imagenet",in_channels=3,classes=1).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE=="cuda"))

def masked_loss(logits,y):
    valid=y!=IGNORE
    return nn.functional.binary_cross_entropy_with_logits(logits[valid],y[valid].float())

def eval_loader(dl):
    model.eval(); yy=[]; pp=[]
    with torch.inference_mode():
        for x,y in dl:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",enabled=(DEVICE=="cuda")): p=torch.sigmoid(model(x).squeeze(1))
            v=y!=IGNORE; yy.append(y[v].cpu().numpy()); pp.append(p[v].float().cpu().numpy())
    yy=np.concatenate(yy); pp=np.concatenate(pp)
    return yy,pp

best=-1; wait=0; patience=6; hist=[]
for epoch in range(1,31):
    model.train(); losses=[]; t=time.time()
    for x,y in train_dl:
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=(DEVICE=="cuda")):
            logits=model(x).squeeze(1); loss=masked_loss(logits,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(loss.item())
    yv,pv=eval_loader(val_dl); auc=roc_auc_score(yv,pv); ap=average_precision_score(yv,pv)
    hist.append({"epoch":epoch,"loss":np.mean(losses),"val_auc":auc,"val_ap":ap,"seconds":time.time()-t})
    print(f"epoch {epoch:02d} loss={np.mean(losses):.4f} val_AUC={auc:.4f} val_AP={ap:.4f} time={time.time()-t:.1f}s")
    if ap>best: best=ap; wait=0; torch.save(model.state_dict(),MODEL_DIR/"MOS_1_2_unet_resnet18_best.pt")
    else:
        wait+=1
        if wait>=patience: break
pd.DataFrame(hist).to_csv(EVAL_DIR/"training_history.csv",index=False)
model.load_state_dict(torch.load(MODEL_DIR/"MOS_1_2_unet_resnet18_best.pt",map_location=DEVICE))

## 5 — Pixel-level held-out test evaluation

These metrics use only labelled pixels in the spatially held-out test blocks. Threshold-free ROC-AUC and AP are primary; a validation-derived F1 threshold is used only for thresholded Dice/IoU/precision/recall reporting.

In [ ]:
yv,pv=eval_loader(val_dl); yt,pt=eval_loader(test_dl)
prec,rec,thr=precision_recall_curve(yv,pv)
f1=2*prec*rec/(prec+rec+1e-12); j=int(np.nanargmax(f1[:-1])); TH=float(thr[j])
pred=(pt>=TH).astype(np.uint8); tn,fp,fn,tp=confusion_matrix(yt,pred,labels=[0,1]).ravel()
pixel_metrics={
 "n_test_pixels":int(len(yt)),"roc_auc":float(roc_auc_score(yt,pt)),"average_precision":float(average_precision_score(yt,pt)),
 "threshold_from_validation":TH,"precision":float(tp/(tp+fp+1e-12)),"recall":float(tp/(tp+fn+1e-12)),
 "specificity":float(tn/(tn+fp+1e-12)),"dice_f1":float(2*tp/(2*tp+fp+fn+1e-12)),"iou":float(tp/(tp+fp+fn+1e-12))}
print(json.dumps(pixel_metrics,indent=2)); (EVAL_DIR/"pixel_test_metrics.json").write_text(json.dumps(pixel_metrics,indent=2))

### 5b — Validation threshold diagnostics

The binary threshold is selected **only on validation pixels** by maximizing F1/Dice. These plots show whether the optimum is sharp or lies on a broad plateau. The continuous probability raster itself is not thresholded.


In [ ]:
# Threshold diagnostics — validation data only
# yv, pv and TH were calculated in the pixel-evaluation cell above.
thresholds=np.linspace(0.01,0.99,199)
rows=[]
for t in thresholds:
    q=(pv>=t).astype(np.uint8)
    tn_,fp_,fn_,tp_=confusion_matrix(yv,q,labels=[0,1]).ravel()
    pr_=tp_/(tp_+fp_+1e-12)
    re_=tp_/(tp_+fn_+1e-12)
    f1_=2*pr_*re_/(pr_+re_+1e-12)
    rows.append((t,pr_,re_,f1_))
td=pd.DataFrame(rows,columns=["threshold","precision","recall","f1_dice"])
td.to_csv(EVAL_DIR/"validation_threshold_diagnostics.csv",index=False)

# 1. Precision and recall vs threshold
fig,ax=plt.subplots(figsize=(7,5))
ax.plot(td.threshold,td.precision,label="Precision")
ax.plot(td.threshold,td.recall,label="Recall")
ax.axvline(TH,linestyle="--",label=f"Selected = {TH:.3f}")
ax.set(xlabel="Musanga probability threshold",ylabel="Score",ylim=(0,1.02),
       title="Validation precision and recall vs threshold")
ax.legend(); fig.tight_layout()
fig.savefig(EVAL_DIR/"validation_precision_recall_vs_threshold.png",dpi=180)
plt.show()

# 2. F1/Dice vs threshold
fig,ax=plt.subplots(figsize=(7,5))
ax.plot(td.threshold,td.f1_dice,label="F1 / Dice")
ax.axvline(TH,linestyle="--",label=f"Maximum validation F1 = {TH:.3f}")
ax.scatter([TH],[td.iloc[(td.threshold-TH).abs().argmin()].f1_dice],zorder=3)
ax.set(xlabel="Musanga probability threshold",ylabel="F1 / Dice",ylim=(0,1.02),
       title="Validation threshold selection")
ax.legend(); fig.tight_layout()
fig.savefig(EVAL_DIR/"validation_f1_vs_threshold.png",dpi=180)
plt.show()

# 3. Validation precision-recall curve with selected operating point
pr_curve,rc_curve,thr_curve=precision_recall_curve(yv,pv)
j=np.argmin(np.abs(thr_curve-TH))
fig,ax=plt.subplots(figsize=(6,5))
ax.plot(rc_curve,pr_curve,label=f"Validation AP = {average_precision_score(yv,pv):.3f}")
ax.scatter([rc_curve[j]],[pr_curve[j]],s=55,
           label=f"Selected threshold = {TH:.3f}")
ax.set(xlabel="Recall",ylabel="Precision",xlim=(0,1.02),ylim=(0,1.02),
       title="Validation precision–recall curve")
ax.legend(); fig.tight_layout()
fig.savefig(EVAL_DIR/"validation_precision_recall_curve.png",dpi=180)
plt.show()

print("Selected validation threshold:",TH)
print("Saved threshold diagnostics to:",EVAL_DIR)

## 6 — Full-mosaic pixel probability inference

Efficient halo-tiled inference writes a georeferenced uint16 probability raster (0–10000; nodata 65535). Each 512×512 core is predicted with 64 px context, avoiding the old crown-window classifier entirely.

In [ ]:
if not LOCAL_MOSAIC.exists() or LOCAL_MOSAIC.stat().st_size!=MOSAIC.stat().st_size:
    print("Copying mosaic to local SSD for inference..."); shutil.copy2(MOSAIC,LOCAL_MOSAIC)
OUT_TIF=PROB_DIR/"MOS_1_2_musanga_pixel_probability_uint16.tif"
CORE=512; HALO=64
model.eval()
with rasterio.open(LOCAL_MOSAIC) as src:
    profile=src.profile.copy(); profile.update(count=1,dtype="uint16",nodata=65535,compress="deflate",predictor=2,tiled=True,BIGTIFF="YES")
    with rasterio.open(OUT_TIF,"w",**profile) as dst:
        jobs=[(r,c,min(CORE,H-r),min(CORE,W-c)) for r in range(0,H,CORE) for c in range(0,W,CORE)]
        for r,c,hh,ww in tqdm(jobs,desc="U-Net full mosaic"):
            win=Window(c-HALO,r-HALO,ww+2*HALO,hh+2*HALO)
            arr=src.read([1,2,3],window=win,boundless=True,fill_value=0).astype(np.float32)/255.
            # pad to multiple of 32 for U-Net
            ph=(32-arr.shape[1]%32)%32; pw=(32-arr.shape[2]%32)%32
            arr=np.pad(arr,((0,0),(0,ph),(0,pw)))
            x=torch.from_numpy((arr-MEAN)/STD).unsqueeze(0).to(DEVICE)
            with torch.inference_mode(), torch.amp.autocast("cuda",enabled=(DEVICE=="cuda")):
                p=torch.sigmoid(model(x).squeeze()).float().cpu().numpy()
            core=p[HALO:HALO+hh,HALO:HALO+ww]
            enc=np.clip(np.round(core*10000),0,10000).astype(np.uint16)
            dst.write(enc,1,window=Window(c,r,ww,hh))
print("Saved:",OUT_TIF)

## 7 — Crown-level held-out evaluation

The **same pixel probability raster** is now aggregated within each held-out test crown. No crown classifier is trained. Mean pixel probability is the primary crown score; median and high quantile are also saved for inspection.

In [ ]:
test_crowns=ann[ann.split=="test"].copy()
rows=[]
with rasterio.open(OUT_TIF) as src:
    for row in tqdm(test_crowns.itertuples(),total=len(test_crowns),desc="Crown evaluation"):
        geom=row.geometry; win=rasterio.windows.from_bounds(*geom.bounds,transform=src.transform).round_offsets().round_lengths()
        a=src.read(1,window=win).astype(np.float32); tr=src.window_transform(win)
        inside=rasterize([(geom,1)],out_shape=a.shape,transform=tr,fill=0,dtype="uint8").astype(bool)
        v=a[inside]; v=v[v!=65535]/10000.
        if len(v): rows.append({"crown_id":row.crown_id,"class_id":row.class_id_num,"truth":row.target,"n_pixels":len(v),"prob_mean":float(v.mean()),"prob_median":float(np.median(v)),"prob_p90":float(np.quantile(v,.9))})
crown=pd.DataFrame(rows); crown.to_csv(EVAL_DIR/"crown_test_probabilities.csv",index=False)
yc=crown.truth.values; pc=crown.prob_mean.values
crown_metrics={"n_test_crowns":int(len(crown)),"n_musanga":int(yc.sum()),"n_other":int((1-yc).sum()),"roc_auc_mean_probability":float(roc_auc_score(yc,pc)),"average_precision_mean_probability":float(average_precision_score(yc,pc))}
print(json.dumps(crown_metrics,indent=2)); (EVAL_DIR/"crown_test_metrics.json").write_text(json.dumps(crown_metrics,indent=2))

fig,ax=plt.subplots(figsize=(6,5)); fpr,tpr,_=roc_curve(yc,pc); ax.plot(fpr,tpr,label=f"Crown AUC={crown_metrics['roc_auc_mean_probability']:.3f}"); ax.plot([0,1],[0,1],"--"); ax.set(xlabel="False positive rate",ylabel="True positive rate",title="Held-out crown ROC"); ax.legend(); fig.tight_layout(); fig.savefig(EVAL_DIR/"crown_roc.png",dpi=180); plt.show()

## 8 — Summary

The outputs to carry forward are the trained U-Net checkpoint, the full-resolution pixel probability GeoTIFF, and the independent pixel- and crown-level test metrics. No probability calibration is imposed at this stage; the raw sigmoid probability surface is retained. Cross-mosaic calibration/generalization can be tested once additional mosaics are available.

In [ ]:
summary={"mosaic":str(MOSAIC),"annotations":str(ANNOT),"model":"U-Net / ResNet18 ImageNet encoder","label_rule":"Class_ID 1 = Musanga; other classified crowns = non-Musanga; unlabeled pixels ignored","spatial_block_m":BLOCK_M,"patch_px":PATCH,"pixel_metrics":pixel_metrics,"crown_metrics":crown_metrics,"probability_raster":str(OUT_TIF)}
(OUT/"KVO_01_MOS_1_2_pixel_unet_summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2)); print("\nDONE — UAV pixel probability + pixel/crown evaluation only.")